In [39]:
import pandas as pd
import numpy as np

In [40]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [41]:
data = pd.read_csv(r'D:\Machine Learning\Column Transformer\covid_toy.csv')

In [42]:
data.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [43]:
data.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [44]:
x= data.drop('has_covid',axis=1)
y = data['has_covid']

In [45]:
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.2)
X_train.head()

,age,gender,fever,cough,city
96,51,Female,101.0,Strong,Kolkata
16,69,Female,103.0,Mild,Kolkata
8,19,Female,100.0,Strong,Bangalore
30,15,Male,101.0,Mild,Delhi
55,81,Female,101.0,Mild,Mumbai


## Without Column Transformer

In [46]:
# It is used to fill blank row with mean
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.transform(X_test[['fever']])
X_train_fever.shape

(80, 1)

In [47]:
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [48]:
ohe = OneHotEncoder(sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])
X_test_gender_city = ohe.transform(X_test[['gender','city']])
X_train_gender_city.shape

(80, 6)

In [49]:
X_train_age = X_train['age'].values
X_test_age = X_test['age'].values

In [50]:
X_train_transformed = np.column_stack(
    (
        X_train_fever,
        X_train_cough,
        X_train_gender_city,
        X_train_age
    )
)
X_test_transformed = np.column_stack(
    (
        X_test_age,
        X_test_fever,
        X_test_gender_city,
        X_test_cough
    )
)
X_train_transformed.shape

(80, 9)

In [57]:
X_train_transformed[:5]

array([[101.,   1.,   1.,   0.,   0.,   0.,   1.,   0.,  51.],
       [103.,   0.,   1.,   0.,   0.,   0.,   1.,   0.,  69.],
       [100.,   1.,   1.,   0.,   1.,   0.,   0.,   0.,  19.],
       [101.,   0.,   0.,   1.,   0.,   1.,   0.,   0.,  15.],
       [101.,   0.,   1.,   0.,   0.,   0.,   0.,   1.,  81.]])

## With Column Transformer

In [52]:
from sklearn.compose import ColumnTransformer

In [53]:
transform = ColumnTransformer(transformers=[
    ('tns1',SimpleImputer(),['fever']),
    ('tns2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tns3',OneHotEncoder(sparse_output=False),['gender','city'])
], remainder='passthrough')

In [56]:
transform.fit_transform(X_train)[:5]

array([[101.,   1.,   1.,   0.,   0.,   0.,   1.,   0.,  51.],
       [103.,   0.,   1.,   0.,   0.,   0.,   1.,   0.,  69.],
       [100.,   1.,   1.,   0.,   1.,   0.,   0.,   0.,  19.],
       [101.,   0.,   0.,   1.,   0.,   1.,   0.,   0.,  15.],
       [101.,   0.,   1.,   0.,   0.,   0.,   0.,   1.,  81.]])